# LLM Power Law - Benchmarking Framework on Google Colab

This notebook helps you run LLM benchmarking experiments on Google Colab with free GPU access.

**Hardware:** Colab provides ~15GB VRAM (T4 GPU) - enough for models up to 13B with 4-bit quantization!

## Quick Start

1. **Enable GPU**: Runtime → Change runtime type → GPU (T4)
2. **Run all cells** in order
3. **Monitor progress** with built-in progress bars
4. **Download results** from the Files panel (left sidebar)

## 1. Setup Environment

In [ ]:
# Check GPU availability
!nvidia-smi

import torch
print(f"\n✅ PyTorch version: {torch.__version__}")
print(f"✅ CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"✅ GPU: {torch.cuda.get_device_name(0)}")
    print(f"✅ VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")

In [ ]:
# Clone repository (or use your own)
!git clone https://github.com/YOUR_USERNAME/LLMPowerLaw.git
%cd LLMPowerLaw

In [ ]:
# Install dependencies
print("📦 Installing core dependencies...")
!pip install -q torch transformers accelerate datasets

print("📦 Installing utilities...")
!pip install -q tqdm pyyaml python-dotenv pandas numpy scikit-learn

print("📦 Installing quantization support (4-bit models)...")
!pip install -q bitsandbytes

print("📦 Installing optional packages...")
!pip install -q sentencepiece --only-binary :all:

print("\n✅ Installation complete!")

## 2. Configure Models & Datasets

Colab's free T4 GPU (~15GB VRAM) can run:
- ✅ 7B models with 4-bit quantization (~4GB VRAM)
- ✅ 13B models with 4-bit quantization (~7GB VRAM)
- ✅ Multiple small models (2-3B)

**Pre-configured options below** - just uncomment what you want to test!

In [ ]:
# View current configuration
!cat config/models.yaml | head -n 100

In [ ]:
# Quick configuration for Colab (15GB VRAM)
# This enables models that work well on Colab's free tier

import yaml

# Read current config
with open('config/models.yaml', 'r', encoding='utf-8') as f:
    config = yaml.safe_load(f)

# Enable recommended models for Colab
models_to_enable = [
    'tinyllama-test',      # Fast testing (1.1B)
    'gemma-2b-4bit',       # Excellent quality (2B)
    'phi-3-mini-4bit',     # Balanced (3.8B)
    'llama-2-7b-4bit',     # High quality (7B)
]

# Optionally enable for full Colab power (comment out if testing)
# models_to_enable.append('llama-2-13b-4bit')  # Needs ~7GB VRAM

for model in config['models']:
    if model['name'] in models_to_enable:
        model['enabled'] = True
        print(f"✅ Enabled: {model['name']}")
    else:
        model['enabled'] = False

# Save config
with open('config/models.yaml', 'w', encoding='utf-8') as f:
    yaml.dump(config, f, default_flow_style=False, allow_unicode=True)

print("\n✅ Model configuration updated!")

In [ ]:
# Configure datasets - start with small test
import yaml

with open('config/datasets.yaml', 'r', encoding='utf-8') as f:
    config = yaml.safe_load(f)

# Enable test dataset (5 samples - fast verification)
for dataset in config['datasets']:
    if dataset['name'] == 'custom_classification_test':
        dataset['enabled'] = True
        print(f"✅ Enabled: {dataset['name']} ({dataset.get('num_samples', 'all')} samples)")
    else:
        dataset['enabled'] = False

# Save
with open('config/datasets.yaml', 'w', encoding='utf-8') as f:
    yaml.dump(config, f, default_flow_style=False, allow_unicode=True)

print("\n✅ Dataset configuration updated for quick test!")
print("\n💡 After test succeeds, increase num_samples or enable more datasets")

## 3. Run Benchmark

This will:
1. Load each enabled model
2. Run predictions on test dataset
3. Show progress bars
4. Save results to `results/` folder

In [ ]:
# Run quick test (5 samples)
!python experiments/run_benchmark.py

## 4. Scale Up (After Test Succeeds)

Once the quick test works, increase sample size for real experiments:

In [ ]:
# Scale up to 100 samples
import yaml

with open('config/datasets.yaml', 'r', encoding='utf-8') as f:
    config = yaml.safe_load(f)

for dataset in config['datasets']:
    # Disable test, enable full datasets
    if dataset['name'] == 'custom_classification_test':
        dataset['enabled'] = False
    elif dataset['name'] in ['sst2', 'mnli']:
        dataset['enabled'] = True
        dataset['num_samples'] = 100  # Start with 100, increase as needed
        print(f"✅ Enabled: {dataset['name']} (100 samples)")

with open('config/datasets.yaml', 'w', encoding='utf-8') as f:
    yaml.dump(config, f, default_flow_style=False, allow_unicode=True)

print("\n✅ Scaled up to 100 samples per dataset!")

In [ ]:
# Run full experiment (will take longer)
!python experiments/run_benchmark.py

## 5. View Results

In [ ]:
# List all result files
!ls -lh results/

In [ ]:
# View latest summary
import json
import glob
from pathlib import Path

# Find latest summary file
summary_files = sorted(glob.glob('results/*_summary.json'))
if summary_files:
    latest = summary_files[-1]
    print(f"📊 Reading: {latest}\n")
    
    with open(latest, 'r', encoding='utf-8') as f:
        results = json.load(f)
    
    print(f"Experiment: {results['experiment_name']}")
    print(f"Total experiments: {len(results['experiments'])}")
    print("\nResults:")
    print("=" * 70)
    
    for exp in results['experiments']:
        if exp['status'] == 'completed':
            model = exp['model']
            dataset = exp['dataset']
            metrics = exp.get('metrics', {})
            accuracy = metrics.get('accuracy', 'N/A')
            print(f"✅ {model:<25} | {dataset:<15} | Accuracy: {accuracy}")
        else:
            print(f"❌ {exp['model']:<25} | {exp['dataset']:<15} | Failed")
else:
    print("No results found. Run the benchmark first!")

In [ ]:
# Create a zip file for easy download
!zip -r results.zip results/
print("\n✅ Results zipped! Download 'results.zip' from the Files panel (left sidebar)")

## 6. Advanced: Try Larger Models

Colab has enough VRAM for 13B models with 4-bit quantization:

In [ ]:
# Enable Llama 2 13B (uses ~7GB VRAM)
import yaml

with open('config/models.yaml', 'r', encoding='utf-8') as f:
    config = yaml.safe_load(f)

for model in config['models']:
    if model['name'] == 'llama-2-13b-4bit':
        model['enabled'] = True
        print(f"✅ Enabled: 13B model (expect ~3-5 min load time)")

with open('config/models.yaml', 'w', encoding='utf-8') as f:
    yaml.dump(config, f, default_flow_style=False, allow_unicode=True)

print("\n⚠️  This will be slower but higher quality!")

## 💡 Tips for Colab

1. **Free tier limits**: 12-hour sessions, may disconnect if idle
2. **Save often**: Run with small samples first, then scale up
3. **Download results**: Files are deleted when session ends
4. **Monitor GPU**: Run `!nvidia-smi` in a cell to check usage
5. **Reduce samples**: If timeout, reduce `num_samples` in datasets

## 🚀 Next Steps

- Try different prompting techniques (see `config/prompting_techniques.yaml`)
- Add custom datasets (see `data_loaders/data/`)
- Compare multiple models on same dataset
- Export results and analyze locally